# Defining global variables

In [1]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 3

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [3]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

Cloning into 'Textual_Analysis_in_Finance'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 145 (delta 35), reused 134 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 5.88 MiB | 25.38 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/kaggle/working/Textual_Analysis_in_Finance


# Analyse Tesla's 10-K files in 2024 and 2025

## Objective

10-K files are famous of their **strong persistence**, meaning the 10-K files of a company in two consecutive years **hardly different**. Let's check this fact with Tesla's 10-K files in 2024 and 2025.

Our plan to do it includes the following steps:

1. Read the files
2. Split each file into paragraphs, we will look at the files at the paragraph level. That means we will find the paragraphs in 2025 that resemble those in the 2024 file
3. Organize the paragraphs in a dataframe
4. Find the paragraph embeddings. This can be done in the following steps:
   * Train a Word2Vec model
   * For each paragraph, derive the embedding of words in the paragraph
   * Take the average of word embeddings to get the embedding of the paragraph
   * Repeat for all paragraphs
5. Compute the mutual cosine similarity between paragraphs in the 2025 and 2024 files

## Get yourself familiar with the data

* The file name

In [4]:
import os

file_2024 = os.path.join(BASE_DIR, 'Data', 'FinancialStatements', '20240129_10-K_edgar_data_1318605_0001628280-24-002390.txt')
file_2025 = os.path.join(BASE_DIR, 'Data', 'FinancialStatements', '20250130_10-K_edgar_data_1318605_0001628280-25-003063.txt')

########## Questions ##########
'''
1. Can you explain the file name? E.g., 20240129_10-K_edgar_data_1318605_0001628280-24-002390.txt
'''

'\n1. Can you explain the file name? E.g., 20240129_10-K_edgar_data_1318605_0001628280-24-002390.txt\n'

* Read the content

In [5]:
# Read the 2024 file
with open(file_2024, 'r') as f:
    content_2024 = f.read()
print(content_2024[:5_000])

# Read the 2025 file
with open(file_2025, 'r') as f:
    content_2025 = f.read()

<Header>
<FileStats>
    <FileName>20240129_10-K_edgar_data_1318605_0001628280-24-002390.txt</FileName>
    <GrossFileSize>15527801</GrossFileSize>
    <NetFileSize>360322</NetFileSize>
    <NonText_DocumentType_Chars>2733010</NonText_DocumentType_Chars>
    <HTML_Chars>5923363</HTML_Chars>
    <XBRL_Chars>3328064</XBRL_Chars>
    <XML_Chars>2858062</XML_Chars>
    <N_Exhibits>11</N_Exhibits>
</FileStats>
<SEC-Header>
0001628280-24-002390.hdr.sgml : 20240129
<ACCEPTANCE-DATETIME>20240126210020
ACCESSION NUMBER:		0001628280-24-002390
CONFORMED SUBMISSION TYPE:	10-K
PUBLIC DOCUMENT COUNT:		119
CONFORMED PERIOD OF REPORT:	20231231
FILED AS OF DATE:		20240129
DATE AS OF CHANGE:		20240126

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Tesla, Inc.
		CENTRAL INDEX KEY:			0001318605
		STANDARD INDUSTRIAL CLASSIFICATION:	MOTOR VEHICLES & PASSENGER CAR BODIES [3711]
		ORGANIZATION NAME:           	04 Manufacturing
		IRS NUMBER:				912197729
		STATE OF INCORPORATION:			DE
		FISCAL YEAR END:

In [ ]:
########## Questions ##########
'''
2. Assume words are those separated by a space ' '. Check how many words are there in the above 10-K?
3. Assume paragraphs are those separated by a double line break '\n\n'. Check how many paragraphs are there in the above 10-K?
'''

## Organize the data

We aim to assess the similarity between two 10-K filings by identifying which paragraphs in the 2025 filing closely resemble those in the 2024 filing.

To begin, we split each 10-K document into individual paragraphs and organize them into a dataframe as follows:

| year | para_idx | text       |
|------|----------|------------|
| 2024 | 0        | _text_0_   |
| 2024 | 1        | _text_1_   |
| ...  | ...      | ...        |
| 2025 | 0        | _text_0_   |
| 2025 | 1        | _text_1_   |
| ...  | ...      | ...        |

In [6]:
import pandas as pd

paras_2024 = [(2024, i, para) for i, para in enumerate(content_2024.split('\n\n'))]
paras_2025 = [(2025, i, para) for i, para in enumerate(content_2025.split('\n\n'))]

data = pd.DataFrame(
    data = paras_2024 + paras_2025,
    columns = ['year', 'para_idx', 'text']
)
data

,year,para_idx,text
0,2024,0,<Header>\n<FileStats>\n <FileName>20240129_...
1,2024,1,FILER:
2,2024,2,\tCOMPANY DATA:\t\n\t\tCOMPANY CONFORMED NAME:...
3,2024,3,\tFILING VALUES:\n\t\tFORM TYPE:\t\t10-K\n\t\t...
4,2024,4,\tBUSINESS ADDRESS:\t\n\t\tSTREET 1:\t\t3500 D...
...,...,...,...
425,2025,226,<EX-101.LAB>\n 13\n tsla-20241231_lab.xml\n XB...
426,2025,227,</EX-101.LAB>
427,2025,228,<EX-101.PRE>\n 14\n tsla-20241231_pre.xml\n XB...
428,2025,229,</EX-101.PRE>


## Clean the data

As you can see, the raw 10-K file is very noisy (even after cleaning the HTML format). Let's normalize the data with some remarks:

* Keep the period and double line breaks

In [7]:
from Week_1.Code.Utils import lowercase, special_character_removal, fix_contraction, stopword_removal, lemmatization

def text_normalization(text):
    text = lowercase(text)
    text = fix_contraction(text)
    text = special_character_removal(text, keep = '.', remove_number = True)
    text = stopword_removal(text, remove_single_letters = True)
    text = lemmatization(text)
    return text

data['clean_text'] = data['text'].apply(text_normalization)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Let's train a Word2Vec model using those clean texts

This can be done using the `gensim` package. First, let's import it by `from gensim.models import Word2Vec`.

An example usage is as follows,

```
model = Word2Vec(
    sentences = corpus      # Your tokenized corpus
    vector_size = 100,      # Dimensionality of the word vectors
    window = 5,             # Context window size
    min_count = 2,          # Ignore words with total frequency < 2
    workers = 4,            # Number of CPU cores to use
    epochs = 5,             # Number of iterations the model goes over the corpus
    sg = 1                  # 1 for Skip-gram; 0 for CBOW
    
)
```


#### Prepare the corpus

The `corpus` argument should look like

```
corpus = [
    ['word_1,1', 'word_1,2', ..., 'word_1,1t'],
    ['word_2,1', 'word_2,2', ..., 'word_2,2t'],
    ...,
    ['word_N,1', 'word_N,2', ..., 'word_N,Nt'],
]
```

In [ ]:
from tqdm.auto import tqdm

corpus = '''YOUR CODE HERE'''

# Check the first 3 paragraphs
corpus[:3]

In [8]:
from tqdm.auto import tqdm
from nltk import word_tokenize

# Design the corpus from our data frame
corpus = [word_tokenize(i) for i in tqdm(data['clean_text'])]

# Or ...
# corpus = [i.split() for i in tqdm(data['clean_text'])]

# Check the first 3 paragraphs
corpus[:3]

  0%|          | 0/430 [00:00<?, ?it/s]

[['header',
  'filestat',
  'filename',
  'edgar',
  'datum',
  '.txt',
  'filename',
  'grossfilesize',
  'grossfilesize',
  'netfilesize',
  'netfilesize',
  'nontext',
  'documenttype',
  'char',
  'nontext',
  'documenttype',
  'char',
  'html',
  'chars',
  'html',
  'chars',
  'xbrl',
  'chars',
  'xbrl',
  'chars',
  'xml',
  'chars',
  'xml',
  'chars',
  'exhibit',
  'exhibits',
  'filestat',
  'sec',
  'header',
  '.hdr.sgml',
  'acceptance',
  'datetime',
  'accession',
  'number',
  'conform',
  'submission',
  'type',
  'public',
  'document',
  'count',
  'conform',
  'period',
  'report',
  'file',
  'date',
  'date',
  'change'],
 ['filer'],
 ['company',
  'datum',
  'company',
  'conform',
  'name',
  'tesla',
  'inc',
  '.',
  'central',
  'index',
  'key',
  'standard',
  'industrial',
  'classification',
  'motor',
  'vehicle',
  'passenger',
  'car',
  'body',
  'organization',
  'name',
  'manufacture',
  'irs',
  'number',
  'state',
  'incorporation',
  'de',
  

#### Training

In [9]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    sentences = corpus,
    vector_size = 100,
    window = 5,
    min_count = 2,
    workers = 4,
    sg = 1,
    epochs = 5
)

#### Evaluating the results

* First, check the vocabulary

In [10]:
vocab = word2vec_model.wv.key_to_index
print(f'There are {len(vocab)} words in the vocabulary')

There are 3119 words in the vocabulary


* Check the vector of a single words

In [11]:
word = 'profit'
if word in vocab:
    print(f'The embedding of "{word}" is:')
    print(word2vec_model.wv['profit'])
else:
    print('Your chosen word does not appear in the vocabulary, please choose another word!')

The embedding of "profit" is:
[ 0.38609982  0.11089402 -0.3089412   0.20431231  0.10306477 -0.27479184
  0.29946905  0.17973694 -0.03523283 -0.28923622 -0.07818075 -0.24826397
 -0.12890808 -0.41941208 -0.03646769  0.22112583  0.07324852 -0.15793477
 -0.52560824 -0.5505714  -0.09591822  0.33152127  0.03490964 -0.25195417
  0.05752752 -0.04948274 -0.23867124 -0.07541491 -0.3676326   0.02372116
 -0.1011913   0.02164989  0.11563727 -0.2596358  -0.02590085  0.31759667
 -0.1471896   0.01549087 -0.32542387 -0.36282182 -0.07866179  0.03811721
 -0.01694796  0.30355707  0.18931052 -0.5421964  -0.01391817 -0.04233297
  0.21189176  0.49761525 -0.06926291 -0.34648496  0.39095157 -0.01290733
  0.11072219  0.4992187  -0.1025903   0.3636112  -0.46902204  0.16250274
  0.17915332  0.05149984  0.18668044  0.14486139 -0.41755757  0.03520496
  0.11201648  0.4987226  -0.24261129  0.27438614 -0.00694379  0.24659021
  0.32184616 -0.13011718  0.03766579 -0.19036849  0.07712992  0.02770237
  0.08535448 -0.01464

In [12]:
word2vec_model.wv.most_similar('profit')

[('gross', 0.9755398035049438),
 ('margin', 0.9694795608520508),
 ('segment', 0.9566613435745239),
 ('above', 0.9556031823158264),
 ('reportable', 0.9555156826972961),
 ('amortization', 0.954131007194519),
 ('upfront', 0.9451067447662354),
 ('consist', 0.9439374804496765),
 ('type', 0.943549394607544),
 ('comprise', 0.9416274428367615)]

## Obtain paragraph embeddings

So far, with Word2Vec,we converted words to vectors (embedding). However, we want to find the similarity between paragraphs in the 2025 and 2024 filings. Consequently, we need a way to convert paragraphs (not words) to vectors as well.

A simple way is to take the **average of the word embeddings** in the corresponding paragraph.

First, let's start with one sample. The implementation includes the following steps:

1. Given a sample text, **tokenize** it in a similar way to what we've done with `corpus`
2. **Loop** over the words in the text
3. For each word, **check** if the word is in the vocabulary or not
4. If not, **continue** the loop
5. If yes, **retrieve** the embedding of that word
6. **Sum** the embedding across words, then **divide** it by the total number of words in the sample text

In [13]:
import numpy as np

sample_para = data['clean_text'].iloc[0]

para_embedding = np.zeros((len(sample_para.split()), 100))

for i, word in enumerate(sample_para.split()):
    if word not in word2vec_model.wv:
        continue
    else:
        para_embedding[i] = word2vec_model.wv[word]
para_embedding = np.mean(para_embedding, axis = 0)
para_embedding

array([-0.1303838 ,  0.01126722, -0.05275777, -0.13974855, -0.08518707,
       -0.19525525,  0.08614184,  0.36500958, -0.10845137, -0.08251722,
       -0.18020533, -0.26889236, -0.11069426, -0.21698534,  0.0009456 ,
       -0.03544448,  0.06540293, -0.25630709, -0.08490369, -0.06760414,
        0.05129902,  0.0872805 ,  0.23146011, -0.05216144,  0.01137118,
       -0.09266868, -0.07258864, -0.1605309 , -0.0091766 ,  0.05314947,
        0.00944753,  0.07051482,  0.0425976 , -0.04214371, -0.08600347,
        0.27452485,  0.00718675, -0.22380274, -0.10034969, -0.31025046,
       -0.08394899, -0.09326829,  0.04517918, -0.01704049,  0.09057768,
       -0.06505292,  0.04262611,  0.06871684, -0.03420314,  0.09203681,
       -0.0521002 , -0.03447303,  0.0630598 ,  0.05527032,  0.02213182,
       -0.11365543,  0.13938244,  0.10119794, -0.27337717, -0.00389744,
        0.00412282,  0.096964  , -0.06008648, -0.04371506, -0.15269213,
        0.01826865,  0.0335119 ,  0.05192678, -0.02963082,  0.12

* **Your turn:** wrap this in a function and apply for all texts

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

def sentence_embedding(text):
    para_embedding = ...
    '''YOUR CODE HERE'''
    return pd.Series(para_embedding)

In [14]:
from tqdm.auto import tqdm
tqdm.pandas()

def sentence_embedding_word2vec(text):
    para_embedding = np.zeros((len(text.split()), 100))
    for i, word in enumerate(text.split()):
        if word not in word2vec_model.wv:
            continue
        else:
            para_embedding[i] = word2vec_model.wv[word]
    para_embedding = np.mean(para_embedding, axis = 0)
    return pd.Series(para_embedding)

para_embedding = data['clean_text'].progress_apply(sentence_embedding_word2vec)
para_embedding

  0%|          | 0/430 [00:00<?, ?it/s]

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.130384,0.011267,-0.052758,-0.139749,-0.085187,-0.195255,0.086142,0.365010,-0.108451,-0.082517,...,-0.191975,0.017147,0.028891,0.023802,0.425579,0.309165,0.081065,-0.182649,-0.084926,-0.036960
1,-0.096565,0.047931,-0.143248,-0.159656,-0.067475,-0.143985,0.055502,0.149096,-0.156678,-0.146668,...,-0.266360,0.089951,0.015003,0.026214,0.395049,0.367969,0.051856,-0.153011,-0.031018,-0.022855
2,-0.048670,0.031806,-0.040615,-0.115851,-0.048826,-0.313597,0.110122,0.363630,-0.114220,-0.139112,...,-0.166737,0.102344,0.131283,0.065947,0.486891,0.384188,0.052693,-0.155223,-0.099828,-0.058639
3,-0.039755,-0.094166,-0.110234,-0.250342,-0.044796,-0.180493,0.264241,0.424749,-0.127173,-0.135256,...,-0.422042,-0.066814,0.041955,0.016686,0.667794,0.486631,0.008204,-0.154552,-0.050093,-0.163552
4,-0.128645,0.147353,-0.077462,-0.104919,0.000075,-0.288735,0.076331,0.251257,-0.134032,-0.142072,...,-0.052625,0.080944,0.107086,0.092073,0.335665,0.273060,0.053161,-0.109284,-0.036584,-0.045063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,-0.247103,-0.083010,-0.017637,-0.193019,-0.144332,-0.104247,0.077038,0.445198,-0.071657,0.021658,...,-0.186926,-0.028668,-0.017008,-0.061619,0.447717,0.282435,0.156739,-0.332809,-0.149975,-0.089315
426,-0.203109,-0.062824,0.017658,-0.102059,-0.124532,-0.083887,0.041804,0.380514,-0.007587,0.004037,...,-0.109420,0.007677,0.016144,-0.015159,0.355883,0.250335,0.125742,-0.290519,-0.156986,-0.076475
427,-0.254532,-0.079605,-0.028339,-0.194018,-0.144261,-0.123175,0.070810,0.445079,-0.080256,0.016447,...,-0.203584,-0.028683,-0.018071,-0.058446,0.460685,0.294498,0.162351,-0.333311,-0.141852,-0.086165
428,-0.201296,-0.064753,0.013118,-0.092222,-0.128848,-0.081543,0.053739,0.388108,-0.012365,-0.012360,...,-0.112258,0.011216,0.020917,-0.014466,0.358809,0.249797,0.116498,-0.275399,-0.151101,-0.078372


In [ ]:
########## Questions ##########
'''
4. There is a row with all NaN values, why does this happen? Check how many such rows there are
'''

* Concatenate to the orginial data. Now, we know the embedding of each paragraph, and we also know the information of the paragraph (year and index)

In [15]:
data = pd.concat([data, para_embedding], axis = 1)
data = data.dropna()
data

,year,para_idx,text,clean_text,0,1,2,3,4,5,...,90,91,92,93,94,95,96,97,98,99
0,2024,0,<Header>\n<FileStats>\n <FileName>20240129_...,header filestat filename edgar datum .txt file...,-0.130384,0.011267,-0.052758,-0.139749,-0.085187,-0.195255,...,-0.191975,0.017147,0.028891,0.023802,0.425579,0.309165,0.081065,-0.182649,-0.084926,-0.036960
1,2024,1,FILER:,filer,-0.096565,0.047931,-0.143248,-0.159656,-0.067475,-0.143985,...,-0.266360,0.089951,0.015003,0.026214,0.395049,0.367969,0.051856,-0.153011,-0.031018,-0.022855
2,2024,2,\tCOMPANY DATA:\t\n\t\tCOMPANY CONFORMED NAME:...,company datum company conform name tesla inc ....,-0.048670,0.031806,-0.040615,-0.115851,-0.048826,-0.313597,...,-0.166737,0.102344,0.131283,0.065947,0.486891,0.384188,0.052693,-0.155223,-0.099828,-0.058639
3,2024,3,\tFILING VALUES:\n\t\tFORM TYPE:\t\t10-K\n\t\t...,file value form type sec act act sec file numb...,-0.039755,-0.094166,-0.110234,-0.250342,-0.044796,-0.180493,...,-0.422042,-0.066814,0.041955,0.016686,0.667794,0.486631,0.008204,-0.154552,-0.050093,-0.163552
4,2024,4,\tBUSINESS ADDRESS:\t\n\t\tSTREET 1:\t\t3500 D...,business address street deer creek rd city pal...,-0.128645,0.147353,-0.077462,-0.104919,0.000075,-0.288735,...,-0.052625,0.080944,0.107086,0.092073,0.335665,0.273060,0.053161,-0.109284,-0.036584,-0.045063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,2025,225,</EX-101.DEF>,ex .def,-0.195706,-0.066985,0.006214,-0.088645,-0.120155,-0.084040,...,-0.104387,0.015659,0.028790,-0.004812,0.347571,0.249063,0.108811,-0.267948,-0.140178,-0.076127
425,2025,226,<EX-101.LAB>\n 13\n tsla-20241231_lab.xml\n XB...,ex .lab tsla lab.xml xbrl taxonomy extension l...,-0.247103,-0.083010,-0.017637,-0.193019,-0.144332,-0.104247,...,-0.186926,-0.028668,-0.017008,-0.061619,0.447717,0.282435,0.156739,-0.332809,-0.149975,-0.089315
426,2025,227,</EX-101.LAB>,ex .lab,-0.203109,-0.062824,0.017658,-0.102059,-0.124532,-0.083887,...,-0.109420,0.007677,0.016144,-0.015159,0.355883,0.250335,0.125742,-0.290519,-0.156986,-0.076475
427,2025,228,<EX-101.PRE>\n 14\n tsla-20241231_pre.xml\n XB...,ex .pre tsla pre.xml xbrl taxonomy extension p...,-0.254532,-0.079605,-0.028339,-0.194018,-0.144261,-0.123175,...,-0.203584,-0.028683,-0.018071,-0.058446,0.460685,0.294498,0.162351,-0.333311,-0.141852,-0.086165


#### Compute the cosine similarity using `scipy.spatial.distance.cdist`
([Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html#scipy.spatial.distance.cdist))

_Notice that, this function compute the **distance**, not similarity. To obtain the similarity, we take `1 - distance`_

The result of this step is a matrix, whose the number of rows (columns) is the number of paragraphs in the 2025 (2024) filing.

In [16]:
# Paragraph embeddings in 2024
embedding_2024 = data.loc[data.year == 2024, range(100)]
para_2024 = data.loc[data.year == 2024, ['year', 'para_idx', 'text']]

# Paragraph embeddings in 2025
embedding_2025 = data.loc[data.year == 2025, range(100)]
para_2025 = data.loc[data.year == 2025, ['year', 'para_idx', 'text']]

# Compute the similarity
from scipy.spatial.distance import cdist

similarity = 1 - cdist(embedding_2025, embedding_2024, metric = 'cosine')
similarity = pd.DataFrame(similarity)
similarity

,0,1,2,3,4,5,6,7,8,9,...,186,187,188,189,190,191,192,193,194,195
0,1.000000,0.904927,0.932507,0.917545,0.853534,0.906783,0.912242,0.959580,0.941798,0.798552,...,0.898967,0.859454,0.903140,0.857575,0.908185,0.861489,0.899420,0.853665,0.914596,0.864791
1,0.904927,1.000000,0.890539,0.827725,0.860692,0.893572,0.834699,0.899071,0.915130,0.672649,...,0.763944,0.738851,0.772060,0.742990,0.778638,0.746168,0.765894,0.735000,0.785048,0.743494
2,0.932507,0.890539,1.000000,0.833988,0.917761,0.956845,0.927993,0.874588,0.952272,0.712490,...,0.771531,0.779838,0.774333,0.773653,0.778619,0.777894,0.768271,0.767704,0.782335,0.777934
3,0.917545,0.827725,0.833988,1.000000,0.718365,0.776252,0.804522,0.932021,0.788456,0.600323,...,0.768374,0.680349,0.771301,0.673751,0.778626,0.679789,0.768655,0.671374,0.788432,0.689213
4,0.853534,0.860692,0.917761,0.718365,1.000000,0.973414,0.806618,0.818524,0.939728,0.593613,...,0.658018,0.658672,0.665879,0.657988,0.667908,0.663551,0.658004,0.647396,0.678619,0.658338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,0.861489,0.746168,0.777894,0.679789,0.663551,0.747745,0.831720,0.777295,0.827178,0.987746,...,0.966266,0.998789,0.964189,0.998754,0.961841,1.000000,0.963007,0.998138,0.957894,0.998234
225,0.899420,0.765894,0.768271,0.768655,0.658004,0.750308,0.827826,0.859933,0.824544,0.950076,...,0.999587,0.963825,0.999838,0.966522,0.999536,0.963007,1.000000,0.967381,0.998582,0.969738
226,0.853665,0.735000,0.767704,0.671374,0.647396,0.736012,0.825262,0.776652,0.818672,0.991263,...,0.969997,0.998445,0.967897,0.999226,0.965749,0.998138,0.967381,1.000000,0.961212,0.998369
227,0.914596,0.785048,0.782335,0.788432,0.678619,0.764457,0.838209,0.879488,0.836825,0.939877,...,0.997639,0.957692,0.998620,0.960501,0.999238,0.957894,0.998582,0.961212,1.000000,0.963845


* The value of the entry $(i,j)$ is the cosine similarity between the $i$-th paragraph in 2025 and the $j$-paragraph in 2024

Based on this similarity matrix, we can find the top $n$ most similar paragraphs in the 2024 file to the $i$-th paragraph in the 2025 file. The steps include:

1. Choose the paragraph index in 2025
2. Define `topn`, which is the number of paragraphs that are most similar to the chosen paragraph in 2025
3. Use the method `np.argsort` to find the index of the `topn` similar paragraphs
4. Retrieve the 2024 aragraphs based on the found indexes

In [17]:
i = 75
topn = 5

print('The paragraph in 2025 is:')
print(para_2025.loc[para_2025.para_idx == i, 'text'].item())

print('\n\n')
print(f'The top {topn} most similar paragraphs in the 2024 file are:')

# Find the indexes of the most similar paragraphs
top_similar_para_idx = np.argsort(similarity[i])[::-1][:topn]

# Based on the above indexes, retrieve the paragraphs
for idx in top_similar_para_idx:
    print('*' * 10)
    print(f'(The similarity is {similarity[i][idx]})')
    print(para_2024.loc[para_2024.para_idx == idx, 'text'].item())

The paragraph in 2025 is:
However, we operate in a cyclical industry that is sensitive to shifting consumer trends, political and regulatory uncertainty, including with respect to trade and the environment, all of which can be compounded by inflationary pressures, rising energy prices, interest rate fluctuations and the liquidity of enterprise customers. For example, as inflationary pressures increased across the markets in which we operate, central banks in developed countries raised interest rates rapidly and substantially, which impacted the affordability of vehicle lease and finance arrangements. Further, sales of vehicles in the automotive industry also tend to be cyclical in many markets, which may expose us to increased volatility as we expand and adjust our operations. Moreover, as additional competitors enter the marketplace and help bring the world closer to sustainable transportation, we will have to adjust and continue to execute well to maintain our momentum. Additionally,

In [ ]:
########## Questions ##########
'''
5. What do you think about the result? Is it satisfactory? If not, what are the potential reasons?
'''

# Sentiment analysis with Financial Phrasebank and Doc2Vec

Last week, we performed sentiment analysis using several approaches, including word-count methods based on the Harvard-IV and Loughran–McDonald dictionaries, as well as supervised learning models such as logistic regression with Bag-of-Words and TF-IDF representations.

This week, we extend our analysis by applying Doc2Vec, a neural embedding method. In theory, this approach should yield better results than the previous methods.

We use the Financial PhraseBank dataset. As in last week, we implement the sentiment analysis with the following steps:

1. Text normalization
2. Split the data into 2 subsets, one for training, the other one for testing. We will compare the results of different models using the testing set
3. Use the texts in the training data to **train the Doc2Vec model**
4. Use the Doc2Vec model to extract document embeddings
5. Train the prediction model

## Load, normalize the data, and split the data into two groups

_We reuse the code from last week_

In [18]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

from Week_1.Code.Utils import lowercase, special_character_removal, fix_contraction, stopword_removal, lemmatization

# Load the data
agreement_level = '50'    # There are '50', '66', '75', and 'All' agreement levels
financial_phrasebank = pd.read_csv(
    os.path.join(BASE_DIR, 'Data', 'FinancialPhrasebank', f'Sentences_{agreement_level}Agree.txt'), 
    sep = '@',
    encoding = 'latin-1',
    header = None,
    names = ['text', 'sentiment']
)

# Normalize the text
def text_normalization(text):
    text = lowercase(text)
    text = special_character_removal(text, keep = None, remove_number = True)
    text = fix_contraction(text)
    text = stopword_removal(text, remove_single_letters = True)
    text = lemmatization(text)
    return text

financial_phrasebank['clean_text'] = financial_phrasebank['text'].apply(text_normalization)

# Encode the label
LABEL_ENCODE = {
    'neutral': 0,
    'negative': 1,
    'positive': 2
}
LABEL_ENCODE_INV = {v: k for k, v in LABEL_ENCODE.items()}

financial_phrasebank['sentiment_encoded'] = financial_phrasebank['sentiment'].map(LABEL_ENCODE)

# Split the data
train_data, test_data = train_test_split(
    financial_phrasebank, 
    test_size = 0.5,
    random_state = 1,
    stratify = financial_phrasebank['sentiment_encoded']
)

## Train the Doc2Vec model

To train a Doc2Vec model, we need to prepare the documents, which is more complicated than Word2Vec.

Training a Doc2Vec model includes the following steps:

1. Prepare the corpus (documents) to have the following form
```
corpus = [
    ['word_1,1', 'word_1,2', ..., 'word_1,1t'],
    ['word_2,1', 'word_2,2', ..., 'word_2,2t'],
    ...,
    ['word_N,1', 'word_N,2', ..., 'word_N,Nt'],
]
```

2. Tag the document using the `TaggedDocument` method as follows:

```
from gensim.models.doc2vec import TaggedDocument

tagged_documents = [TaggedDocument(words = words, tags = [str(i)]) for i, words in enumerate(corpus)]
```

3. Initialize the model

```
model = Doc2Vec(
    vector_size = 100,   # Dimensionality of the word vectors
    window = 5,          # Context window size
    min_count = 2,       # Ignore words with total frequency < 2
    workers = 4,         # Number of CPU cores to use in parallel
    epochs = 40,         # Number of iterations the model goes over the corpus
    dm = 1               # 1 for Distributed Memory; 0 for DBOW
)
```

4. Build the vocabulary

```
model.build_vocab(tagged_documents)
```

5. Train

```
model.train(tagged_docs)
```

**Let's put everything together!**

In [19]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

corpus = [word_tokenize(i) for i in train_data['clean_text']]

tagged_documents = [TaggedDocument(words = words, tags = [str(i)]) for i, words in enumerate(corpus)]

doc2vec_model = Doc2Vec(
    vector_size = 100,
    window = 5,
    min_count = 2,
    workers = 4,
    epochs = 40,
    dm = 0
)

doc2vec_model.build_vocab(tagged_documents)

doc2vec_model.train(tagged_documents, total_examples = doc2vec_model.corpus_count, epochs = doc2vec_model.epochs)

If you want to save the model for later use, type `model.save(os.path.join('<path_to_output>', 'doc2vec.model'))`. 

To load a saved model, type `model = Doc2Vec.load(os.path.join('<path_to_output>', 'doc2vec.model'))`

#### Infer document vectors

After training a Doc2Vec model, we will use it to infer document embedding using the `model.infer_vector` method. This method takes a **tokenized document as the input**.

An example for a sample text is:

In [20]:
sample_text = train_data['clean_text'].iloc[0]
doc2vec_model.infer_vector(word_tokenize(sample_text))

array([-1.47452906e-01, -2.09584504e-01,  6.24330342e-01,  7.93199539e-02,
        4.78889830e-02,  1.09005935e-01, -4.79087561e-01,  4.94697422e-01,
        2.32963860e-01, -7.31860846e-02,  9.78961065e-02,  2.01020017e-01,
       -3.39458906e-03, -8.93517062e-02,  1.22156061e-01, -4.02852625e-01,
        4.54567492e-01, -1.25373095e-01,  2.26954997e-01, -1.93301111e-01,
        7.85833374e-02, -1.97889373e-01,  2.73674667e-01, -1.91923946e-01,
        1.77436754e-01,  7.90107176e-02,  5.86623624e-02, -5.02241969e-01,
        1.61238864e-01, -3.13297659e-01,  4.70006138e-01,  2.27908015e-01,
       -1.77667901e-01,  5.18680990e-01, -1.29408598e-01,  1.20232731e-01,
        4.41613793e-02,  2.21322149e-01,  3.14705074e-01,  4.21118140e-01,
       -3.44185531e-01, -6.27923310e-02,  4.64577647e-03, -2.83753872e-01,
       -1.44965589e-01, -2.66537666e-01,  3.80917549e-01,  1.33021966e-01,
       -1.48480535e-01,  5.95281553e-03, -6.30210638e-02,  1.09297961e-01,
        4.08610255e-01, -

**Your turn:** Write a function that can be applied for all texts

In [ ]:
def sentence_embedding(text):
    text_embedding = ...
    '''YOUR CODE HERE'''
    return pd.Series(text_embedding)

In [21]:
def sentence_embedding(text):
    text_embedding = doc2vec_model.infer_vector(word_tokenize(text))
    return pd.Series(text_embedding)

X_train_doc2vec = train_data['clean_text'].progress_apply(sentence_embedding)
X_test_doc2vec = test_data['clean_text'].progress_apply(sentence_embedding)

  0%|          | 0/2423 [00:00<?, ?it/s]

  0%|          | 0/2423 [00:00<?, ?it/s]

#### Extract text vector using Bag-of-Words and TF-IDF

I wrap the code from last week in an object called `TextFeatureExtractor`. To use it, first load it

`from Week_2.Code.Utils import TextFeatureExtractor`

In [22]:
from Week_2.Code.Utils import TextFeatureExtractor

X_train_bow, X_test_bow = TextFeatureExtractor(method = 'bow')(train_data['clean_text'], test_data['clean_text'])
X_train_tfidf, X_test_tfidf = TextFeatureExtractor(method = 'tfidf')(train_data['clean_text'], test_data['clean_text'])

y_train = train_data['sentiment_encoded']
y_test = test_data['sentiment_encoded']

## Train the prediction model

In [23]:
from sklearn.linear_model import LogisticRegression

# Initialize the model
print('Train the Bag-of-Words model')
model_bow = LogisticRegression(penalty = None)
model_bow.fit(X_train_bow, y_train)

print('Train the TF-IDF model')
model_tfidf = LogisticRegression(penalty = None)
model_tfidf.fit(X_train_tfidf, y_train)

print('Train the Doc2Vec model')
model_doc2vec = LogisticRegression(penalty = None)
model_doc2vec.fit(X_train_doc2vec, y_train)

print('Prediction')
test_data['sentiment_pred_bow'] = model_bow.predict(X_test_bow)
test_data['sentiment_pred_tfidf'] = model_tfidf.predict(X_test_tfidf)
test_data['sentiment_pred_doc2vec'] = model_doc2vec.predict(X_test_doc2vec)

print('Encode the prediction')
test_data['sentiment_pred_bow'] = test_data['sentiment_pred_bow'].map(LABEL_ENCODE_INV)
test_data['sentiment_pred_tfidf'] = test_data['sentiment_pred_tfidf'].map(LABEL_ENCODE_INV)
test_data['sentiment_pred_doc2vec'] = test_data['sentiment_pred_doc2vec'].map(LABEL_ENCODE_INV)

Train the Bag-of-Words model
Train the TF-IDF model
Train the Doc2Vec model
Prediction
Encode the prediction


In [24]:
from sklearn.metrics import f1_score

score_bow = f1_score(test_data['sentiment'], test_data['sentiment_pred_bow'], average = 'micro')
score_tfidf = f1_score(test_data['sentiment'], test_data['sentiment_pred_tfidf'], average = 'micro')
score_doc2vec = f1_score(test_data['sentiment'], test_data['sentiment_pred_doc2vec'], average = 'micro')

print('F1 scores:')
print('- Bag-of-Words:', score_bow)
print('- TF-IDF:', score_tfidf)
print('- Doc2Vec:', score_doc2vec)

F1 scores:
- Bag-of-Words: 0.7160544779199339
- TF-IDF: 0.7197688815517953
- Doc2Vec: 0.6838629797771357


In [ ]:
########## Questions ##########
'''
6. What do you think about the result? Is it satisfactory? If not, what are the potential reasons?
'''

# ELMo

Coding ELMo requires some advanced knowledge about Pytorch, including dataset and dataloader design, neural network construction. So, I decide to only provide the code as reference. Feel free to check the code to train an ELMo model in `Week_3.Code.ELMo` in the lecture's Github repo.